# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")
print("Token loaded successfully")

Token loaded successfully


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1) Two paper findings + my methodology questions

### Finding 1 — The Content Performance Curve

The paper reports that content health peaks around 61–90 days and declines after 270 days. It also reports a recovery for older pages that were refreshed.

**Methodology question:** Where exactly does the outcome come from, and how is the refresh effect separated from other factors? Because the study is observational, older pages that were refreshed may differ systematically from pages that were not refreshed. I would therefore want to know whether the comparison controls for differences such as existing visibility, content quality, and search demand before interpreting the refresh pattern as evidence of an effect.

### Finding 2 — The Freshness Multiplier

The paper reports that 365+ day content refreshed within 30 days showed a 3.2× health increase and a 57× impression increase.

**Methodology question:** Does the validation design support interpreting these large differences as a refresh effect? The paper notes that the 365+ freshness bucket can be small and unstable, so I would want to see the sample sizes and a comparable untreated group. A before/after comparison alone could also be affected by regression to the mean or other changes occurring at the same time.

These questions are not a criticism of the findings. They are checks I would apply to understand how strongly the available evidence supports each claim.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2) My model under an honest split

In Week 5, I used a standard train/test split. For this audit, I re-ran the model using a grouped split by client.

The grouped split keeps all content from the same client in either the training set or the test set, rather than allowing the same client to appear in both.

I compare the Week-5 result with the grouped-split result. The grouped result is the more honest estimate of how the model may perform on clients that were not used during training.


In [4]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_Token")
login(token=HF_TOKEN)

content_df = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)["train"].to_pandas()

print("Dataset loaded!")
print("Rows:", len(content_df))
print("Columns:", len(content_df.columns))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

Dataset loaded!
Rows: 519606
Columns: 26


In [5]:
from sklearn.model_selection import GroupShuffleSplit

audit_df = content_df.copy()

audit_df = audit_df.dropna(subset=["client_hash_id"]).copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        audit_df,
        groups=audit_df["client_hash_id"]
    )
)

train_df = audit_df.iloc[train_idx].copy()
test_df = audit_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print("Client overlap:", len(overlap))

Train rows: 439038
Test rows: 80568
Train clients: 67
Test clients: 17
Client overlap: 0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3) Leakage audit

I audited the final feature set for information leakage.

I checked whether any feature directly contains the target, is derived from the target, or uses information from the outcome period. I also checked the time windows of performance features.

For a refresh-opportunity model, features from the outcome period should not be used to predict that same outcome. Features such as `trend_pct` and `trend_direction` are especially risky because they are directly related to the decline label.

The final feature set should therefore use only information that would have been available before the prediction period.


In [6]:
# Section 3: Leakage audit

# Columns that are directly related to the decline label
label_related = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

print("Checking label-related columns:\n")

for col in label_related:
    if col in content_df.columns:
        print("FOUND:", col)
    else:
        print("Not present:", col)

print("\nPotential future/outcome columns:")

future_keywords = [
    "last30",
    "current",
    "future",
    "outcome"
]

for col in content_df.columns:
    if any(word in col.lower() for word in future_keywords):
        print("-", col)

Checking label-related columns:

Not present: is_declining_label
Not present: trend_direction
Not present: trend_pct

Potential future/outcome columns:


In [7]:
# Check for obvious ID columns being used as features

id_columns = [
    col for col in content_df.columns
    if "id" in col.lower() or "hash" in col.lower()
]

print("Potential ID columns:")
for col in id_columns:
    print("-", col)

Potential ID columns:
- client_hash_id
- content_hash_id
- keyword_hash_id
- url_hash_id
- provider_used


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4) Claim rewrite

**Original claim:**

My model can identify the pages that should be refreshed and can predict which content will perform better after a refresh.

**Safer claim:**

My analysis observed patterns in the available content and performance data that can help prioritize pages for review. The model provides directional, decision-support signals for identifying potential refresh opportunities, but it does not establish that refreshing a page will cause better performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.